In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [7]:
import os

img_dir = Path("../data/raw/VOCtrainval-2007/JPEGImages")
img_list = os.listdir(img_dir)

annot_dir = Path("../data/raw/VOCtrainval-2007/Annotations")
annot_list = os.listdir(annot_dir)

img_list[:5], annot_list[:5]

(['000005.jpg', '000007.jpg', '000009.jpg', '000012.jpg', '000016.jpg'],
 ['000005.xml', '000007.xml', '000009.xml', '000012.xml', '000016.xml'])

### Cleaner XML Parsing

In [3]:
import xml.etree.ElementTree as ET

tree = ET.parse(os.path.join(annot_dir, annot_list[0]))
root = tree.getroot()

root.tag, root.attrib

('annotation', {})

In [4]:
filename = root.find("filename").text
filename

'000005.jpg'

In [5]:
from src.configs import IMAGE_SIZE

size = root.find("size")
width = int(size.find("width").text)
height = int(size.find("height").text)
img_width_ratio = IMAGE_SIZE / width
img_height_ratio = IMAGE_SIZE / height

print(width, height, img_width_ratio, img_height_ratio)

500 375 0.448 0.5973333333333334


In [32]:
import numpy as np

objects = []

for obj in root.findall("object"):
    class_name = obj.find("name").text
    
    bndbox = obj.find("bndbox")
    xmin = int(bndbox.find("xmin").text) * img_width_ratio
    ymin = int(bndbox.find("ymin").text) * img_height_ratio
    xmax = int(bndbox.find("xmax").text) * img_width_ratio
    ymax = int(bndbox.find("ymax").text) * img_height_ratio

    center = (np.mean([xmin, xmax]), np.mean([ymin, ymax]))
    width = (xmax - xmin)
    height = (ymax - ymin) 
    
    objects.append((class_name, center[0], center[1], width, height))
    
objects

[('chair',
  np.float64(131.488),
  np.float64(164.26666666666668),
  27.328000000000017,
  76.45866666666666),
 ('chair',
  np.float64(93.632),
  np.float64(189.95200000000003),
  39.42400000000001,
  64.512),
 ('chair',
  np.float64(16.128),
  np.float64(184.57600000000002),
  27.776000000000003,
  77.65333333333334),
 ('chair',
  np.float64(120.064),
  np.float64(147.24266666666668),
  24.191999999999993,
  62.72000000000001),
 ('chair',
  np.float64(131.936),
  np.float64(121.25866666666668),
  15.680000000000007,
  20.309333333333342)]

### Visualize Pandas Read CSV Dataframe

In [42]:
import pandas as pd

annotations_file = Path("../data/preprocessed/trainval/annotations.csv")
df = pd.read_csv(annotations_file)

df

,filename,class_name,x,y,w,h
0,000005.jpg,chair,131,164,27,76
1,000005.jpg,chair,94,190,39,65
2,000005.jpg,chair,16,185,28,78
3,000005.jpg,chair,120,147,24,63
4,000005.jpg,chair,132,121,16,20
...,...,...,...,...,...,...
15657,009958.jpg,person,53,82,27,128
15658,009958.jpg,person,50,55,32,57
15659,009958.jpg,bicycle,57,147,45,131
15660,009959.jpg,car,117,98,62,27


In [43]:
df.iloc[0].filename

'000005.jpg'

In [44]:
groups = df.groupby("filename")

In [46]:
for filename, group in groups:
    print(filename, group)

000005.jpg      filename class_name    x    y   w   h
0  000005.jpg      chair  131  164  27  76
1  000005.jpg      chair   94  190  39  65
2  000005.jpg      chair   16  185  28  78
3  000005.jpg      chair  120  147  24  63
4  000005.jpg      chair  132  121  16  20
000007.jpg      filename class_name    x    y    w    h
5  000007.jpg        car  144  128  161  188
000009.jpg      filename class_name    x    y   w   h
6  000009.jpg      horse   76  150  90  94
7  000009.jpg     person   85  127  35  85
8  000009.jpg     person  137  159  19  78
9  000009.jpg     person  124  157  17  78
000012.jpg       filename class_name    x    y   w    h
10  000012.jpg        car  114  123  87  116
000016.jpg       filename class_name    x    y    w    h
11  000016.jpg    bicycle  133  122  143  180
000017.jpg       filename class_name    x    y    w    h
12  000017.jpg     person  108   80   44   84
13  000017.jpg      horse  115  127  146  159
000019.jpg       filename class_name    x    y    w

In [50]:
filenames = list(df["filename"].unique())
filenames

['000005.jpg',
 '000007.jpg',
 '000009.jpg',
 '000012.jpg',
 '000016.jpg',
 '000017.jpg',
 '000019.jpg',
 '000020.jpg',
 '000021.jpg',
 '000023.jpg',
 '000024.jpg',
 '000026.jpg',
 '000030.jpg',
 '000032.jpg',
 '000033.jpg',
 '000034.jpg',
 '000035.jpg',
 '000036.jpg',
 '000039.jpg',
 '000041.jpg',
 '000042.jpg',
 '000044.jpg',
 '000046.jpg',
 '000047.jpg',
 '000048.jpg',
 '000050.jpg',
 '000051.jpg',
 '000052.jpg',
 '000060.jpg',
 '000061.jpg',
 '000063.jpg',
 '000064.jpg',
 '000065.jpg',
 '000066.jpg',
 '000072.jpg',
 '000073.jpg',
 '000077.jpg',
 '000078.jpg',
 '000081.jpg',
 '000083.jpg',
 '000089.jpg',
 '000091.jpg',
 '000093.jpg',
 '000095.jpg',
 '000099.jpg',
 '000101.jpg',
 '000102.jpg',
 '000104.jpg',
 '000107.jpg',
 '000109.jpg',
 '000110.jpg',
 '000112.jpg',
 '000113.jpg',
 '000117.jpg',
 '000118.jpg',
 '000120.jpg',
 '000121.jpg',
 '000122.jpg',
 '000123.jpg',
 '000125.jpg',
 '000129.jpg',
 '000130.jpg',
 '000131.jpg',
 '000132.jpg',
 '000133.jpg',
 '000134.jpg',
 '000138.j

In [59]:
group = groups.get_group(filenames[0]).drop(columns=["filename"])
group

,class_name,x,y,w,h
0,chair,131,164,27,76
1,chair,94,190,39,65
2,chair,16,185,28,78
3,chair,120,147,24,63
4,chair,132,121,16,20


In [106]:
for i in group.iterrows():
    print(tuple(i[1]))

('chair', 131, 164, 27, 76)
('chair', 94, 190, 39, 65)
('chair', 16, 185, 28, 78)
('chair', 120, 147, 24, 63)
('chair', 132, 121, 16, 20)


In [66]:
len(group)

5

In [65]:
tuple(group.iloc[0])

('chair', np.int64(131), np.int64(164), np.int64(27), np.int64(76))

In [96]:
def get_labels(groups, filename):
    group = groups.get_group(filename).drop(columns=["filename"])

    objects = []
    for object_ in group:
        objects.append(tuple(object_))
    
    return objects

In [97]:
get_labels(groups, filenames[0])

[('c', 'l', 'a', 's', 's', '_', 'n', 'a', 'm', 'e'),
 ('x',),
 ('y',),
 ('w',),
 ('h',)]

### Test Dataset

In [87]:
from torchvision.io import decode_image
from torch.utils.data import Dataset

class ImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        img_df = pd.read_csv(annotations_file)
        self.filenames = list(img_df["filename"].unique()) # list of image file names
        self.groups = img_df.groupby("filename") # filename groups contain respective objects

        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_filename = self.filenames[idx]

        img_path = img_dir / img_filename
        image = decode_image(img_path)

        labels = get_labels(self.groups, img_filename)

        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            labels = self.target_transform(labels)

        return image, labels

In [88]:
annotations_file = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/Images")
dataset = ImageDataset(annotations_file, img_dir)

In [91]:
image, label = list(iter(dataset))[0]

In [93]:
image.shape, label

(torch.Size([3, 224, 224]),
 [('chair', np.int64(131), np.int64(164), np.int64(27), np.int64(76)),
  ('chair', np.int64(94), np.int64(190), np.int64(39), np.int64(65)),
  ('chair', np.int64(16), np.int64(185), np.int64(28), np.int64(78)),
  ('chair', np.int64(120), np.int64(147), np.int64(24), np.int64(63)),
  ('chair', np.int64(132), np.int64(121), np.int64(16), np.int64(20))])